# OCR de PDFs com `labdados`

<a href="https://colab.research.google.com/github/lab-dados/labdados-sdk/blob/main/examples/notebooks/ocr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este notebook mostra como extrair texto de PDFs com o SDK `labdados` em **dois modos**:

1. **Nuvem** — usa o serviço do escritório (precisa de API key).
2. **Local** — roda Tesseract no próprio Colab (sem chave, mais lento).

Cada célula de código é independente: você pode rodar só a do modo nuvem, só a do modo local, ou as duas. As células de instalação e de baixar o PDF de exemplo são pré-requisitos compartilhados.

📘 Documentação completa: <https://lab-dados.github.io/labdados-sdk>

## 1. Instalar

Para o **modo nuvem**, basta o pacote core (sem extras). Para o **modo local**, instalamos o extra `[ocr]` e o binário do Tesseract.

In [ ]:
# Modo nuvem (mínimo)
%pip install -q labdados

In [ ]:
# Modo local — extras + binário Tesseract (idiomas pt + en)
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por tesseract-ocr-eng
%pip install -q 'labdados[ocr]'

## 2. Baixar um PDF de exemplo

Pegamos um paper público hospedado pelo Mozilla pdf.js ('TraceMonkey', ~14 páginas em inglês). No seu uso real, troque por sua própria pasta com PDFs.

In [ ]:
import os

os.makedirs('meus_pdfs', exist_ok=True)
!wget -q -O meus_pdfs/tracemonkey.pdf https://mozilla.github.io/pdf.js/web/compressed.tracemonkey-pldi-09.pdf
!ls -la meus_pdfs/

## 3. Modo nuvem (com API key)

Pegue sua chave no portal do escritório:
<https://labdados-frontend.livelydesert-3e3e3dd8.brazilsouth.azurecontainerapps.io/consultoria/api-key>.
Cole quando o `getpass` perguntar — ela **não fica visível** no notebook.

In [ ]:
import os
from getpass import getpass

import labdados

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

labdados.ocr(
    arquivos='meus_pdfs/',
    api_key=os.environ['LABDADOS_API_KEY'],
    saida='resultado_nuvem/',
    formato='md',          # 'txt' ou 'md'
    idiomas='por+eng',
)
!ls -la resultado_nuvem/

O resultado vem em `.zip` com um arquivo de texto por PDF. Para descompactar e inspecionar:

In [ ]:
!cd resultado_nuvem && unzip -o ocr_*.zip && ls
!head -30 resultado_nuvem/tracemonkey.md 2>/dev/null || head -30 resultado_nuvem/tracemonkey.txt

## 4. Modo local (sem API key)

Para 5–10 PDFs leves, dá pra rodar sem custo nenhum direto no runtime do Colab. **Pré-requisito**: a célula de instalação do extra `[ocr]` + Tesseract acima precisa ter rodado.

Para layouts complexos (várias colunas, formulários, scans ruidosos), troque para `modelo='paddleocr'` — funciona melhor mas a primeira chamada demora ~30s para subir.

In [ ]:
import labdados

labdados.ocr(
    arquivos='meus_pdfs/tracemonkey.pdf',
    local=True,
    idiomas='por+eng',
    formato='md',
    saida='resultado_local/',
    deskew=True,            # endireita scans tortos
)
!ls -la resultado_local/
!head -30 resultado_local/tracemonkey.md

## 5. Próximos passos

- Para uma **pasta inteira** com centenas de PDFs, use `arquivos='minha_pasta/'` (varre subpastas).
- Para reusar a mesma chave em várias chamadas, use `cli = labdados.Client(api_key=...)`.
- Para encadear OCR + extração estruturada, veja o [notebook de estruturação](https://colab.research.google.com/github/lab-dados/labdados-sdk/blob/main/examples/notebooks/estruturacao.ipynb).

Documentação dos parâmetros: <https://lab-dados.github.io/labdados-sdk/api/ocr.html>